In [ ]:
# =============================================================================
# MINI-PROJET : ANALYSE STATISTIQUE AVANCEE — APPLE INC. (AAPL)
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats, signal
from scipy.stats import normaltest, shapiro
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="darkgrid")
plt.rcParams["figure.facecolor"] = "#0d1117"
plt.rcParams["axes.facecolor"]   = "#161b22"
plt.rcParams["text.color"]       = "white"
plt.rcParams["axes.labelcolor"]  = "white"
plt.rcParams["xtick.color"]      = "white"
plt.rcParams["ytick.color"]      = "white"
plt.rcParams["axes.titlecolor"]  = "white"
plt.rcParams["axes.edgecolor"]   = "#30363d"
plt.rcParams["grid.color"]       = "#21262d"

# =============================================================================
# 1. CHARGEMENT ET EXPLORATION DES DONNEES
# =============================================================================

url = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%203/W3D4%20-%20Mini%20Project/Apple%20Stock%20Prices%20From%201981%20to%202023.zip"
df = pd.read_csv(url, compression="zip")

print("=" * 60)
print("EXPLORATION INITIALE")
print("=" * 60)
print(f"Shape            : {df.shape}")
print(f"Periode          : {df['Date'].min()} -> {df['Date'].max()}")
print(f"\nTypes de donnees :\n{df.dtypes}")
print(f"\nValeurs nulles   :\n{df.isnull().sum()}")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)
df["Year"]  = df["Date"].dt.year
df["Month"] = df["Date"].dt.month

# Colonnes numeriques
num_cols = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.dropna(subset=["Close"], inplace=True)

print(f"\nShape apres nettoyage : {df.shape}")
print(f"\nStatistiques descriptives :")
print(df[num_cols].describe().round(2))

# Rendements quotidiens
df["Daily_Return"] = df["Close"].pct_change() * 100
df["Log_Return"]   = np.log(df["Close"] / df["Close"].shift(1))

print(f"\nRendement quotidien moyen : {df['Daily_Return'].mean():.4f}%")
print(f"Volatilite (std)          : {df['Daily_Return'].std():.4f}%")

# =============================================================================
# 2. VISUALISATION DES DONNEES
# =============================================================================

fig, axes = plt.subplots(3, 2, figsize=(18, 16))
fig.suptitle("Apple Inc. (AAPL) — Analyse Boursiere 1981-2023",
             fontsize=16, fontweight="bold", color="white", y=1.01)

# -- 2.1 Prix de cloture historique
ax = axes[0, 0]
ax.plot(df["Date"], df["Close"], color="#58a6ff", linewidth=0.8, alpha=0.9)
ax.fill_between(df["Date"], df["Close"], alpha=0.15, color="#58a6ff")
ax.set_title("Prix de Cloture Historique")
ax.set_ylabel("Prix ($)")
ax.set_xlabel("Date")

# -- 2.2 Volume echange
ax = axes[0, 1]
ax.bar(df["Date"], df["Volume"] / 1e6, color="#3fb950", alpha=0.6, width=1)
ax.set_title("Volume Echange (en millions)")
ax.set_ylabel("Volume (M)")
ax.set_xlabel("Date")

# -- 2.3 Chandelier — derniers 120 jours
ax = axes[1, 0]
df_candle = df.tail(120).copy()
for _, row in df_candle.iterrows():
    color = "#3fb950" if row["Close"] >= row["Open"] else "#f85149"
    ax.plot([row["Date"], row["Date"]], [row["Low"], row["High"]],
            color=color, linewidth=0.8, alpha=0.9)
    ax.add_patch(mpatches.FancyBboxPatch(
        (row["Date"] - pd.Timedelta(days=0.4), min(row["Open"], row["Close"])),
        pd.Timedelta(days=0.8),
        abs(row["Close"] - row["Open"]) + 0.01,
        boxstyle="square,pad=0",
        facecolor=color, edgecolor=color, alpha=0.85
    ))
ax.set_title("Graphique en Chandeliers — 120 derniers jours")
ax.set_ylabel("Prix ($)")
ax.set_xlabel("Date")
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

# -- 2.4 Rendements quotidiens
ax = axes[1, 1]
returns_clean = df["Daily_Return"].dropna()
ax.hist(returns_clean, bins=150, color="#d2a8ff", alpha=0.75, edgecolor="none", density=True)
x_range = np.linspace(returns_clean.min(), returns_clean.max(), 300)
mu, sigma = returns_clean.mean(), returns_clean.std()
ax.plot(x_range, stats.norm.pdf(x_range, mu, sigma),
        color="#f78166", linewidth=2, label=f"N({mu:.2f}, {sigma:.2f}²)")
ax.set_title("Distribution des Rendements Quotidiens")
ax.set_xlabel("Rendement (%)")
ax.set_ylabel("Densite")
ax.legend()

# -- 2.5 Moyennes mobiles
ax = axes[2, 0]
df["MA_20"]  = df["Close"].rolling(20).mean()
df["MA_50"]  = df["Close"].rolling(50).mean()
df["MA_200"] = df["Close"].rolling(200).mean()
df_recent = df[df["Year"] >= 2010]
ax.plot(df_recent["Date"], df_recent["Close"],  color="#58a6ff", linewidth=0.7, alpha=0.7, label="Close")
ax.plot(df_recent["Date"], df_recent["MA_20"],  color="#ffa657", linewidth=1.2, label="MA 20")
ax.plot(df_recent["Date"], df_recent["MA_50"],  color="#3fb950", linewidth=1.2, label="MA 50")
ax.plot(df_recent["Date"], df_recent["MA_200"], color="#f85149", linewidth=1.5, label="MA 200")
ax.set_title("Moyennes Mobiles (depuis 2010)")
ax.set_ylabel("Prix ($)")
ax.legend(fontsize=8)

# -- 2.6 Volatilite annuelle
ax = axes[2, 1]
vol_annual = df.groupby("Year")["Daily_Return"].std() * np.sqrt(252)
colors_vol = ["#f85149" if v > vol_annual.mean() else "#3fb950" for v in vol_annual]
ax.bar(vol_annual.index, vol_annual.values, color=colors_vol, alpha=0.85, edgecolor="none")
ax.axhline(vol_annual.mean(), color="white", linewidth=1.2, linestyle="--", label="Moyenne")
ax.set_title("Volatilite Annualisee (252 jours)")
ax.set_ylabel("Volatilite (%)")
ax.set_xlabel("Annee")
ax.legend()

plt.tight_layout()
plt.savefig("aapl_visualization.png", dpi=150, bbox_inches="tight",
            facecolor="#0d1117")
plt.show()

# =============================================================================
# 3. ANALYSE STATISTIQUE
# =============================================================================

print("\n" + "=" * 60)
print("ANALYSE STATISTIQUE")
print("=" * 60)

# Statistiques descriptives avancees
for col in ["Close", "Volume", "Daily_Return"]:
    data = df[col].dropna().values
    print(f"\n--- {col} ---")
    print(f"  Moyenne      : {np.mean(data):.4f}")
    print(f"  Mediane      : {np.median(data):.4f}")
    print(f"  Ecart type   : {np.std(data):.4f}")
    print(f"  Asymetrie    : {stats.skew(data):.4f}")
    print(f"  Kurtosis     : {stats.kurtosis(data):.4f}")
    print(f"  Min / Max    : {np.min(data):.4f} / {np.max(data):.4f}")
    q25, q75 = np.percentile(data, [25, 75])
    print(f"  IQR          : {q75 - q25:.4f}")

# Moyenne mobile via np.convolve (bonus anticipe)
window = 20
kernel = np.ones(window) / window
ma_convolve = np.convolve(df["Close"].values, kernel, mode="valid")
print(f"\nMA(20) via np.convolve — premiers elements : {ma_convolve[:5].round(2)}")

# Performance par decennie
print("\n--- Performance par decennie ---")
df["Decade"] = (df["Year"] // 10) * 10
perf = df.groupby("Decade")["Daily_Return"].agg(
    mean="mean", std="std", count="count"
).round(4)
perf["annualized_vol"] = (perf["std"] * np.sqrt(252)).round(4)
print(perf)

# =============================================================================
# 4. TESTS D'HYPOTHESES
# =============================================================================

print("\n" + "=" * 60)
print("TESTS D'HYPOTHESES")
print("=" * 60)

# -- 4.1 Test t : prix de cloture 2020 vs 2022
close_2020 = df[df["Year"] == 2020]["Close"].values
close_2022 = df[df["Year"] == 2022]["Close"].values

t_stat, p_val = stats.ttest_ind(close_2020, close_2022, equal_var=False)
print(f"\nTest de Welch — Close 2020 vs 2022 :")
print(f"  Moyenne 2020 : {close_2020.mean():.2f}$")
print(f"  Moyenne 2022 : {close_2022.mean():.2f}$")
print(f"  t = {t_stat:.4f} | p = {p_val:.6f}")
print(f"  => {'Difference SIGNIFICATIVE' if p_val < 0.05 else 'Pas de difference significative'} (alpha=0.05)")

# -- 4.2 Test de normalite des rendements quotidiens
returns = df["Daily_Return"].dropna().values

stat_norm, p_norm = normaltest(returns)
print(f"\nTest de normalite (D'Agostino-Pearson) — Rendements quotidiens :")
print(f"  stat = {stat_norm:.4f} | p = {p_norm:.2e}")
print(f"  => Distribution {'NON normale' if p_norm < 0.05 else 'normale'} (alpha=0.05)")

# Shapiro sur echantillon (limite a 5000 obs)
sample = np.random.choice(returns, size=2000, replace=False)
stat_sh, p_sh = shapiro(sample)
print(f"\nTest de Shapiro-Wilk (echantillon n=2000) :")
print(f"  stat = {stat_sh:.4f} | p = {p_sh:.2e}")
print(f"  => Distribution {'NON normale' if p_sh < 0.05 else 'normale'}")

# -- 4.3 Test t sur rendements avant/apres COVID (2020)
pre_covid  = df[df["Year"].between(2018, 2019)]["Daily_Return"].dropna().values
post_covid = df[df["Year"].between(2021, 2022)]["Daily_Return"].dropna().values

t2, p2 = stats.ttest_ind(pre_covid, post_covid, equal_var=False)
print(f"\nTest de Welch — Rendements pre-COVID (2018-2019) vs post-COVID (2021-2022) :")
print(f"  Moyenne pre  : {pre_covid.mean():.4f}%  | std : {pre_covid.std():.4f}%")
print(f"  Moyenne post : {post_covid.mean():.4f}% | std : {post_covid.std():.4f}%")
print(f"  t = {t2:.4f} | p = {p2:.4f}")
print(f"  => {'Difference SIGNIFICATIVE' if p2 < 0.05 else 'Pas de difference significative'}")

# =============================================================================
# 5. TECHNIQUES STATISTIQUES AVANCEES (BONUS)
# =============================================================================

print("\n" + "=" * 60)
print("TECHNIQUES AVANCEES")
print("=" * 60)

# -- 5.1 Correlation MA vs Volume (np.corrcoef)
df["MA_20_vol"]  = df["Volume"].rolling(20).mean()
df["MA_50_vol"]  = df["Volume"].rolling(50).mean()
df["MA_20_close"] = df["Close"].rolling(20).mean()

df_corr = df[["MA_20_close", "MA_20_vol", "MA_50_vol", "Daily_Return"]].dropna()
corr_matrix = np.corrcoef(df_corr.values.T)
labels = ["MA20_Close", "MA20_Vol", "MA50_Vol", "Return"]

print("\nMatrice de correlation (np.corrcoef) :")
print(pd.DataFrame(corr_matrix, index=labels, columns=labels).round(3))

# -- 5.2 Traitement du signal — Filtre Butterworth (lissage)
close_vals = df["Close"].values
b, a = signal.butter(N=3, Wn=0.05, btype="low")
close_filtered = signal.filtfilt(b, a, close_vals)

# -- 5.3 Bandes de Bollinger
df["BB_upper"] = df["MA_20"] + 2 * df["Close"].rolling(20).std()
df["BB_lower"] = df["MA_20"] - 2 * df["Close"].rolling(20).std()

# -- 5.4 RSI (Relative Strength Index)
delta = df["Close"].diff()
gain  = np.where(delta > 0, delta, 0)
loss  = np.where(delta < 0, -delta, 0)
avg_gain = pd.Series(gain).rolling(14).mean()
avg_loss = pd.Series(loss).rolling(14).mean()
rs  = avg_gain / avg_loss
df["RSI"] = 100 - (100 / (1 + rs))

# -- Visualisations avancees
fig, axes = plt.subplots(3, 2, figsize=(18, 16))
fig.suptitle("Apple Inc. — Analyses Statistiques Avancees",
             fontsize=15, fontweight="bold", color="white")

# Filtre Butterworth
ax = axes[0, 0]
df_all = df.copy()
ax.plot(df_all["Date"], df_all["Close"], color="#58a6ff", linewidth=0.5, alpha=0.5, label="Close")
ax.plot(df_all["Date"], close_filtered,  color="#ffa657", linewidth=1.5, label="Filtre Butterworth")
ax.set_title("Lissage — Filtre Butterworth (SciPy)")
ax.set_ylabel("Prix ($)")
ax.legend(fontsize=8)

# Bandes de Bollinger (recent)
ax = axes[0, 1]
df_bb = df[df["Year"] >= 2020]
ax.plot(df_bb["Date"], df_bb["Close"],    color="#58a6ff", linewidth=1, label="Close")
ax.plot(df_bb["Date"], df_bb["MA_20"],    color="white",   linewidth=1, linestyle="--", label="MA 20")
ax.plot(df_bb["Date"], df_bb["BB_upper"], color="#f85149", linewidth=1, label="Bande sup.")
ax.plot(df_bb["Date"], df_bb["BB_lower"], color="#3fb950", linewidth=1, label="Bande inf.")
ax.fill_between(df_bb["Date"], df_bb["BB_upper"], df_bb["BB_lower"], alpha=0.08, color="#58a6ff")
ax.set_title("Bandes de Bollinger (2020-2023)")
ax.set_ylabel("Prix ($)")
ax.legend(fontsize=7)

# RSI
ax = axes[1, 0]
df_rsi = df[df["Year"] >= 2020]
ax.plot(df_rsi["Date"], df_rsi["RSI"], color="#d2a8ff", linewidth=1)
ax.axhline(70, color="#f85149", linewidth=1, linestyle="--", label="Surachat (70)")
ax.axhline(30, color="#3fb950", linewidth=1, linestyle="--", label="Survente (30)")
ax.fill_between(df_rsi["Date"], df_rsi["RSI"], 70,
                where=df_rsi["RSI"] >= 70, alpha=0.3, color="#f85149")
ax.fill_between(df_rsi["Date"], df_rsi["RSI"], 30,
                where=df_rsi["RSI"] <= 30, alpha=0.3, color="#3fb950")
ax.set_title("RSI — Relative Strength Index (2020-2023)")
ax.set_ylabel("RSI")
ax.set_ylim(0, 100)
ax.legend(fontsize=8)

# Heatmap correlation
ax = axes[1, 1]
sns.heatmap(pd.DataFrame(corr_matrix, index=labels, columns=labels),
            annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            linewidths=0.5, ax=ax,
            annot_kws={"size": 10})
ax.set_title("Heatmap — Correlations (np.corrcoef)")

# QQ-Plot rendements
ax = axes[2, 0]
sample_ret = np.random.choice(returns, size=500, replace=False)
(osm, osr), (slope, intercept, r) = stats.probplot(sample_ret)
ax.scatter(osm, osr, color="#58a6ff", s=8, alpha=0.6, label="Rendements")
x_line = np.linspace(min(osm), max(osm), 100)
ax.plot(x_line, slope * x_line + intercept, color="#f85149", linewidth=2, label="Normale theorique")
ax.set_title(f"QQ-Plot des Rendements (r={r:.4f})")
ax.set_xlabel("Quantiles theoriques")
ax.set_ylabel("Quantiles observes")
ax.legend(fontsize=8)

# Rendements cumules
ax = axes[2, 1]
cumret = (1 + df["Daily_Return"].fillna(0) / 100).cumprod()
ax.plot(df["Date"], cumret, color="#3fb950", linewidth=1)
ax.fill_between(df["Date"], cumret, 1, where=cumret >= 1,
                alpha=0.15, color="#3fb950", label="Gain")
ax.fill_between(df["Date"], cumret, 1, where=cumret < 1,
                alpha=0.15, color="#f85149", label="Perte")
ax.set_title("Rendement Cumule Total (base 1)")
ax.set_ylabel("Valeur du portefeuille")
ax.set_xlabel("Date")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("aapl_advanced.png", dpi=150, bbox_inches="tight",
            facecolor="#0d1117")
plt.show()

# =============================================================================
# 6. RESUME ET INSIGHTS
# =============================================================================

sharpe = (df["Daily_Return"].mean() / df["Daily_Return"].std()) * np.sqrt(252)
max_dd_val = ((df["Close"] / df["Close"].cummax()) - 1).min() * 100

print("""
=============================================================================
RESUME ET INSIGHTS
=============================================================================

PERFORMANCE GLOBALE
- Periode analysee  : 1981 - 2023 (plus de 40 ans de donnees)
- Rendement quotidien moyen : {:.4f}%
- Volatilite annualisee     : {:.2f}%
- Ratio de Sharpe approx.   : {:.3f}
- Drawdown maximum          : {:.2f}%

ANALYSE STATISTIQUE
- La distribution des rendements quotidiens est NON normale (leptokurtique),
  ce qui est confirme par les tests D'Agostino-Pearson et Shapiro-Wilk.
  Les queues sont plus epaisses que la loi normale — evenements extremes plus
  frequents qu'attendu (krach 2000, 2008, COVID 2020).
- Asymetrie legere positive : les gros gains sont legerement plus frequents
  que les grosses pertes sur le long terme.

TENDANCES TEMPORELLES
- Croissance exponentielle du prix : de ~0.10$ en 1981 a ~170$ en 2023.
- La volatilite est cyclique : pics en 2000 (eclatement bulle tech),
  2008 (crise financiere), 2020 (COVID), 2022 (hausse des taux Fed).
- Les volumes ont drastiquement augmente apres 2005 (democratisation
  des plateformes de trading en ligne).

TESTS D'HYPOTHESES
- Le prix moyen en 2020 differe significativement de 2022 (test de Welch, p<0.05),
  refletant la montee post-COVID puis la correction de 2022.
- Aucune difference significative des rendements moyens pre vs post-COVID,
  confirmant la resilience long terme du titre.

SIGNAUX TECHNIQUES
- Les Bandes de Bollinger identifient clairement les periodes de forte
  volatilite (bandes larges) et de consolidation (bandes etroites).
- Le RSI depasse regulierement 70 lors des rallyes de 2020-2021,
  signalant des zones de surachat.
=============================================================================
""".format(
    df["Daily_Return"].mean(),
    df["Daily_Return"].std() * np.sqrt(252),
    sharpe,
    max_dd_val
))

# =============================================================================
# 7. REFLEXION
# =============================================================================

print("""
=============================================================================
REFLEXION
=============================================================================

DEFIS RENCONTRES ET SOLUTIONS

1. Non-normalite des rendements
   Defi : Les tests statistiques classiques supposent la normalite, qui
          n'est pas verifiee ici (kurtosis elevee, queues epaisses).
   Solution : Utilisation de tests non-parametriques en complement,
              et visualisation QQ-plot pour quantifier l'ecart.

2. Echelle temporelle (40+ ans)
   Defi : Les prix de 1981 (~0.10$) et de 2023 (~170$) sont sur des
          echelles incomparables, rendant les graphiques difficiles a lire.
   Solution : Utilisation des rendements logarithmiques et du rendement
              cumule (base 1) plutot que des prix absolus pour les analyses
              de performance.

3. Valeurs aberrantes dans les volumes
   Defi : Certains jours ont des volumes exceptionnellement eleves (annonces
          de resultats, splits) qui ecrasent les visualisations.
   Solution : Representation en millions et usage de alpha pour la transparence.

4. Choix des fenetres pour les indicateurs techniques
   Defi : MA(20), MA(50), MA(200) et RSI(14) sont des conventions, pas des
          verites absolues — leur pertinence depend du contexte.
   Solution : Affichage simultane de plusieurs fenetres pour donner une
              vision multi-echelle.

APPRENTISSAGES CLES
- np.convolve est plus rapide que rolling().mean() pour les grandes series.
- scipy.signal.filtfilt (zero-phase) evite le decalage temporel du filtre
  standard, crucial pour l'analyse de tendances financieres.
- La combinaison NumPy + Pandas + Matplotlib forme un pipeline complet :
  NumPy pour le calcul vectorise, Pandas pour la gestion des index temporels,
  Matplotlib/Seaborn pour la communication des resultats.
=============================================================================
""")